# Lead variant effect

One row per credible set, annotated with MAF in the study's major LD population, a
rescaled effect size, the most severe lead-variant consequence, locus statistics and
variant type. Methods "Annotation of CS lead variants by MAF and beta rescaling".

Writes `lead_variant_effect`.

In [ ]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from pyspark.sql import functions as f

from manuscript_methods import paper
from manuscript_methods.ld_populations import LDPopulationName, LDPopulationStructure
from manuscript_methods.locus_statistics import LocusStatistics
from manuscript_methods.maf import AlleleFrequencies, MinorAlleleFrequency, PopulationFrequency
from manuscript_methods.rescaled_beta import RescaledStatistics
from manuscript_methods.study_statistics import StudyStatistics
from manuscript_methods.tc import LeadVariantConsequences, TranscriptConsequences
from manuscript_methods.variant_statistics import PValueComponents, VariantStatistics
from manuscript_methods.variant_type import Variant

session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

## Credible sets joined to the study and variant indices

In [ ]:
cs = StudyLocus.from_parquet(session, paper.release("credible_set")).df.select(
    "studyId",
    "studyLocusId",
    "variantId",
    "beta",
    "zScore",
    "pValueMantissa",
    "pValueExponent",
    "standardError",
    "finemappingMethod",
    "studyType",
    "locus",
    "isTransQtl",
)
si = StudyIndex.from_parquet(session, paper.release("study")).df.select(
    "studyId",
    "nSamples",
    "nControls",
    "nCases",
    "geneId",
    "diseaseIds",
    "biosampleId",
    "traitFromSourceMappedIds",
    "ldPopulationStructure",
    "traitFromSource",
)
vi = VariantIndex.from_parquet(session, paper.release("variant")).df.select(
    "variantId",
    "alleleFrequencies",
    "variantEffect",
    "transcriptConsequences",
    "chromosome",
    "position",
    "referenceAllele",
    "alternateAllele",
)

dataset = cs.join(si, how="left", on="studyId").join(vi, how="left", on="variantId")

## Major LD population, MAF and allele frequency

In [ ]:
ld_pop = LDPopulationStructure(f.col("ldPopulationStructure"))
major = ld_pop.major_population(default_major_pop=LDPopulationName.NFE)
frequencies = AlleleFrequencies(f.col("alleleFrequencies"))

dataset = dataset.withColumns(
    {
        "majorLdPopulation": major.col,
        "majorLdPopulationMaf": frequencies.ld_population_maf(major.ld_population).col,
        "majorLdPopulationAf": frequencies.ld_population_af(major.ld_population).col,
        "majorLdPopulationsStructure": ld_pop.ld_structure(),
    }
)

## Variant, study and rescaled statistics

In [ ]:
variant_stats = VariantStatistics.compute(
    PValueComponents(p_value_mantissa=f.col("pValueMantissa"), p_value_exponent=f.col("pValueExponent")),
    f.col("nSamples"),
)
study_stats = StudyStatistics.compute(
    n_samples=f.col("nSamples"),
    n_cases=f.col("nCases"),
    n_controls=f.col("nControls"),
    trait=f.col("traitFromSource"),
    study_type=f.col("studyType"),
    is_trans_pqtl=f.col("isTransQtl"),
    gene_id=f.col("geneId"),
)
dataset = dataset.withColumns({"variantStatistics": variant_stats.col, "studyStatistics": study_stats.col})

rescaled = RescaledStatistics.compute(
    beta=f.col("beta"),
    chi2_stat=VariantStatistics(f.col("variantStatistics")).chi2_stat,
    trait_class=StudyStatistics(f.col("studyStatistics")).trait_class,
    af=PopulationFrequency(f.col("majorLdPopulationAf")).allele_frequency,
    maf=MinorAlleleFrequency(f.col("majorLdPopulationMaf")).value,
    n_cases=StudyStatistics(f.col("studyStatistics")).n_cases,
    n_samples=StudyStatistics(f.col("studyStatistics")).n_samples,
)
dataset = dataset.withColumn("rescaledStatistics", rescaled.col)

## Lead variant consequence, locus statistics, variant type

In [ ]:
consequence = LeadVariantConsequences.compute(
    TranscriptConsequences(f.col("transcriptConsequences")),
    StudyStatistics(f.col("studyStatistics")),
)
locus = LocusStatistics.compute(locus=f.col("locus"), lead_variant=f.col("variantId"))
variant = Variant.compute(f.col("chromosome"), f.col("position"), f.col("referenceAllele"), f.col("alternateAllele"))

dataset = dataset.withColumns({consequence.name: consequence.col, locus.name: locus.col, "variant": variant.col})

## Write

In [ ]:
dataset = dataset.select(
    "variantId",
    "variant",
    "studyLocusId",
    "studyId",
    "geneId",
    "diseaseIds",
    "biosampleId",
    f.col("beta").alias("originalBeta"),
    f.col("standardError").alias("originalStandardError"),
    "locusStatistics",
    "finemappingMethod",
    "isTransQtl",
    "variantEffect",
    "majorLdPopulation",
    "majorLdPopulationMaf",
    "majorLdPopulationAf",
    "variantStatistics",
    "studyStatistics",
    "rescaledStatistics",
    "leadVariantConsequence",
    "traitFromSourceMappedIds",
)
dataset.repartition(50).write.mode("overwrite").parquet(paper.derived("lead_variant_effect"))

In [ ]:
written = session.spark.read.parquet(paper.derived("lead_variant_effect"))
print("rows:", written.count())
print("baseline rows:", session.spark.read.parquet(paper.baseline("lead_variant_effect")).count())